# ESANN combined correction notebook (runnable source)

This notebook joins **Experiment 2 — original QR native structural analysis** and **Experiment 4 — fixed hostname-disjoint holdout robustness**. Each section was executed and audited separately with the supplied files (see both `_EXECUTED.ipynb` notebooks). This joined source notebook is **not** claimed to have been rerun from the top as one combined execution; it is a convenience one-file reproduction workflow. Put all input files in the same folder and run from that working directory.

`14-vs-10` paired comparison is in `ESANN_14_vs_10_Feature_Audit_MatchedSeeds_EXECUTED.ipynb`, rather than retraining the frozen model in this notebook.


# ESANN — Corrected Experiments 2 & 4 (original native QR, strict held-out robustness)

This notebook keeps the existing **frozen** final 10-feature model. It was corrected to operate on the original 9,987 Version-13 **69×69 QR matrices** and the unchanged 9,713-row lexical/URL table. The original file with an `.xls` extension is *CSV plain text*; a byte-identical `.csv` copy is used.

**Required inputs (alongside this notebook)**: `qr_codes_29.pickle`, `qr_codes_29_labels.pickle`, `qr_full_feature_dataset_v2.csv`, `FINAL_10feature_XGB_for_deployment.pkl`, `phase14B_final_robust_features.json` and `esann_icann_psl_snapshot.txt` (fallback only if `tldextract` cannot be imported). The PSL snapshot corresponds to system ICANN-only public suffix rules; it has been validated against `num_subdomains` in all 9,713 rows of the provided feature CSV. Its correctness for different URLs / future versions is not established; deploy using compatible `tldextract` where available.

**Experimental controls**: fixed original indices, sample-label consistency assertion, 42-seed grouped holdout, five-fold stratified-group CV, paired hostname bootstrap. The structural features are computed from **native matrices, without QR regeneration, border removal, rescaling or QR-version feature**. Scores are new results, not recycled from previously regenerated QR experiments. Experiment 4 evaluates only the 1,777 unseen-host holdout rows.

The original author manuscripts' Experiment 2 and Experiment 4 numbers must be replaced with newly executed results. Never interpret successful pipeline robustness as evidence of external-domain suitability.

## Experiment 2 — Original QR matrix structural attribution

In [ ]:
import os
import re
import gc
import pickle
import warnings
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse, urlunparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, wilcoxon

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    r2_score,
    mean_absolute_error
)
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBClassifier



warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
N_BOOT = 1000

np.random.seed(SEED)

DATA_DIR = Path(".")

print("Ready.")

In [ ]:
# A hard requirement: do not silently reconstruct a different sample table.
FULL_FEATURE_CSV = DATA_DIR / "qr_full_feature_dataset_v2.csv"
if not FULL_FEATURE_CSV.exists():
    raise FileNotFoundError(f"Required frozen training table not found: {FULL_FEATURE_CSV.resolve()}")
df_existing = pd.read_csv(FULL_FEATURE_CSV)
base_cols = ["original_index", "url", "normalized_url", "hostname", "label"]
assert set(base_cols).issubset(df_existing.columns)
base_df = df_existing[base_cols].copy()
print("Frozen training data:", base_df.shape)


In [ ]:
try:
    import tldextract

    # Avoid unexpected online Public Suffix List download.
    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except Exception:
    domain_extractor = None


SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_lexical_features(url):

    raw_url = str(url).strip()
    lower_url = raw_url.lower()

    scheme_present = int(
        lower_url.startswith(("http://", "https://"))
    )

    parse_url = raw_url

    if not scheme_present:
        parse_url = "http://" + raw_url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""

    url_length = len(raw_url)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    if domain_extractor is not None:

        ext = domain_extractor(hostname)

        if ext.subdomain:
            num_subdomains = len(
                [
                    x
                    for x in ext.subdomain.split(".")
                    if x
                ]
            )
        else:
            num_subdomains = 0

    else:
        # Fallback only if tldextract is unavailable.
        parts = [x for x in hostname.split(".") if x]
        num_subdomains = max(len(parts) - 2, 0)

    suspicious_keyword_count = sum(
        keyword in lower_url
        for keyword in SUSPICIOUS_KEYWORDS
    )

    try:
        contains_port = int(
            parsed.port is not None
        )
    except ValueError:
        contains_port = 0

    return {
        "url_length": url_length,
        "hostname_length": len(hostname),
        "path_length": len(path),
        "query_length": len(query),

        "num_dots": raw_url.count("."),
        "num_hyphens": raw_url.count("-"),
        "num_digits": num_digits,
        "num_special_chars": num_special_chars,

        "digit_ratio":
            num_digits / max(url_length, 1),

        "special_char_ratio":
            num_special_chars / max(url_length, 1),

        "num_subdomains":
            num_subdomains,

        "has_query":
            int(bool(query)),

        "has_fragment":
            int(bool(fragment)),

        "contains_at":
            int("@" in raw_url),

        "contains_port":
            contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:

# Native QR matrices: 0=black, 1=white, no quiet-zone, 69x69 fixed QR Version 13.
# The original sample array is never re-encoded or resized.
def original_binary_matrix(raw):
    raw = np.asarray(raw)
    assert raw.shape == (69,69), f"Expected 69x69, received {raw.shape}"
    assert np.isin(raw, (0,1)).all(), "Expected original binary modules (0 black; 1 white)"
    return (raw == 0).astype(np.uint8)

def _run_lengths(values_2d):
    # Flatten each independent row. Mark each row start explicitly.
    transitions = values_2d[:, 1:] != values_2d[:, :-1]
    starts = np.concatenate([np.ones((values_2d.shape[0],1),bool), transitions],axis=1)
    positions=np.flatnonzero(starts.ravel())
    ends=np.r_[positions[1:], values_2d.size]
    return (ends-positions).astype(np.int16)

def extract_structural_features(matrix):
    m=np.asarray(matrix,dtype=np.uint8)
    assert m.shape == (69,69)
    htrans=(m[:,1:] != m[:,:-1]); vtrans=(m[1:,:] != m[:-1,:])
    # 2x2 bitmap patterns, order TL,TR,BL,BR; 16 possible patterns.
    pattern = (m[:-1,:-1] << 3) | (m[:-1,1:] << 2) | (m[1:,:-1] << 1) | m[1:,1:]
    counts=np.bincount(pattern.ravel(),minlength=16)
    probs=counts[counts>0].astype(float)/counts.sum()
    h_runs=_run_lengths(m); v_runs=_run_lengths(m.T)
    return {
        'black_ratio':float(m.mean()),
        'horizontal_transition_density':float(htrans.mean()),
        'vertical_transition_density':float(vtrans.mean()),
        'overall_transition_density':float((htrans.sum()+vtrans.sum())/(htrans.size+vtrans.size)),
        'pattern_2x2_entropy':float(-(probs*np.log2(probs)).sum()),
        'pattern_2x2_diversity':float(np.count_nonzero(counts)/16),
        'mean_horizontal_run':float(h_runs.mean()),
        'std_horizontal_run':float(h_runs.std()),
        'max_horizontal_run':float(h_runs.max()),
        'mean_vertical_run':float(v_runs.mean()),
        'std_vertical_run':float(v_runs.std()),
        'max_vertical_run':float(v_runs.max())
    }
# Tiny known-pattern self-test, avoids subtly swapped bit conventions.
_t=np.zeros((69,69),dtype=np.uint8)
assert extract_structural_features(_t)['pattern_2x2_entropy']==0.
assert extract_structural_features(_t)['black_ratio']==0.


In [ ]:
# NEVER load cached regenerated structural features.
# Retain existing lexical values to keep the lexical detector definitions fixed.
QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"
with open(QR_FILE, "rb") as f:
    original_qrs = np.asarray(pickle.load(f))
with open(LABEL_FILE, "rb") as f:
    original_labels = np.asarray(pickle.load(f)).astype(int)
assert len(original_qrs) == len(original_labels) == 9987
assert original_qrs.shape[1:] == (69, 69)
assert base_df.original_index.is_unique
indices = base_df.original_index.astype(int).to_numpy()
assert np.all((0 <= indices) & (indices < len(original_qrs)))
assert np.array_equal(base_df.label.astype(int).to_numpy(), original_labels[indices]), "Original-index/label mismatch"

canonical_lexical = ["url_length","hostname_length","path_length","query_length",
 "num_dots","num_hyphens","num_digits","num_special_chars","special_char_ratio",
 "num_subdomains","has_fragment","contains_at","contains_port","suspicious_keyword_count"]
if not set(canonical_lexical).issubset(df_existing.columns):
    raise ValueError("Canonical cached lexical columns are missing; do not silently regenerate under a new parser")

structural_records = []
for k, index in enumerate(indices):
    native_black_1 = original_binary_matrix(original_qrs[index])
    structural_records.append(extract_structural_features(native_black_1))
    if (k + 1) % 1000 == 0: print(f"Native QR features: {k+1}/{len(indices)}")

structural_df = pd.DataFrame(structural_records)
# Empirically constant features are not valid model predictors.
constant_structural = [c for c in structural_df if structural_df[c].nunique(dropna=False) <= 1]
print("Constant original-QR features:", constant_structural)
structural_df = structural_df.drop(columns=constant_structural)
base_columns = ["original_index","url","normalized_url","hostname","label"]
df_all = pd.concat([
    base_df[base_columns].reset_index(drop=True),
    df_existing[canonical_lexical].reset_index(drop=True),
    structural_df.reset_index(drop=True)
], axis=1)
assert len(df_all) == 9713
assert not {"qr_version", "qr_matrix_size"}.intersection(df_all.columns)
print("Corrected data matrix:", df_all.shape)


In [ ]:
LEXICAL_ALL=canonical_lexical.copy()
STRUCTURAL_ALL=list(structural_df.columns)
lexical_reduced=LEXICAL_ALL.copy()
structural_reduced=STRUCTURAL_ALL.copy()
structural_intrinsic=STRUCTURAL_ALL.copy() # only residual diagnostic; no duplicate model family
lexical_plus_structural=lexical_reduced + structural_reduced
print('14 lexical features:',lexical_reduced)
print('Native QR features:',structural_reduced)


In [ ]:
required_cols = list(dict.fromkeys(
    lexical_plus_structural
    +
    ["hostname", "label"]
))

missing_cols = [
    c
    for c in required_cols
    if c not in df_all.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print(
    "Missing feature values:",
    df_all[
        lexical_plus_structural
    ].isna().sum().sum()
)

print(
    "Infinite feature values:",
    np.isinf(
        df_all[
            lexical_plus_structural
        ].astype(float)
    ).sum().sum()
)

print(
    "Class counts:"
)
print(
    df_all["label"].value_counts()
)

print(
    "Unique hostnames:",
    df_all["hostname"].nunique()
)

In [ ]:
# part A -- Signal Attribution
cross_corr_records = []

for lex in lexical_reduced:

    for structural in structural_reduced:

        rho, p = spearmanr(
            df_all[lex],
            df_all[structural],
            nan_policy="omit"
        )

        cross_corr_records.append({
            "lexical_feature": lex,
            "structural_feature": structural,
            "spearman_rho": rho,
            "abs_rho": abs(rho),
            "p_value": p
        })

cross_corr_df = (
    pd.DataFrame(cross_corr_records)
    .sort_values(
        "abs_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top lexical–structural associations:")
display(
    cross_corr_df.head(25).round(4)
)

In [ ]:
cross_matrix = pd.DataFrame(
    index=lexical_reduced,
    columns=structural_reduced,
    dtype=float
)

for lex in lexical_reduced:
    for structural in structural_reduced:
        cross_matrix.loc[lex, structural] = (
            spearmanr(
                df_all[lex],
                df_all[structural],
                nan_policy="omit"
            )[0]
        )

fig, ax = plt.subplots(
    figsize=(12, 8)
)

im = ax.imshow(
    cross_matrix.values.astype(float),
    aspect="auto",
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    np.arange(len(structural_reduced))
)

ax.set_xticklabels(
    structural_reduced,
    rotation=70,
    ha="right"
)

ax.set_yticks(
    np.arange(len(lexical_reduced))
)

ax.set_yticklabels(
    lexical_reduced
)

ax.set_title(
    "Lexical–Structural Spearman Correlations"
)

fig.colorbar(
    im,
    ax=ax,
    label="Spearman rho"
)

plt.tight_layout()
plt.show()

In [ ]:
groups_all = (
    df_all["hostname"]
    .astype(str)
    .reset_index(drop=True)
)

y_label_all = (
    df_all["label"]
    .astype(int)
    .reset_index(drop=True)
)

X_lex_all = (
    df_all[
        lexical_reduced
    ]
    .reset_index(drop=True)
)

structural_predictability_rows = []

attr_cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# Precompute fold assignments once so every target
# uses exactly the same grouped folds.
attr_splits = list(
    attr_cv.split(
        X_lex_all,
        y_label_all,
        groups=groups_all
    )
)

for target in structural_reduced:

    y_target = (
        df_all[target]
        .astype(float)
        .reset_index(drop=True)
    )

    oof_pred = np.full(
        len(df_all),
        np.nan
    )

    for fold, (tr_idx, va_idx) in enumerate(
        attr_splits,
        start=1
    ):

        reg = RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=3,
            random_state=SEED + fold,
            n_jobs=-1
        )

        reg.fit(
            X_lex_all.iloc[tr_idx],
            y_target.iloc[tr_idx]
        )

        oof_pred[va_idx] = reg.predict(
            X_lex_all.iloc[va_idx]
        )

    target_range = (
        y_target.max()
        -
        y_target.min()
    )

    mae = mean_absolute_error(
        y_target,
        oof_pred
    )

    nmae = (
        mae / target_range
        if target_range > 0
        else np.nan
    )

    structural_predictability_rows.append({
        "structural_feature": target,
        "oof_r2_from_lexical":
            r2_score(
                y_target,
                oof_pred
            ),
        "oof_mae":
            mae,
        "normalized_mae":
            nmae
    })

structural_predictability_df = (
    pd.DataFrame(
        structural_predictability_rows
    )
    .sort_values(
        "oof_r2_from_lexical",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    structural_predictability_df.round(4)
)

In [ ]:
predictability_summary = pd.DataFrame([{
    "mean_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].mean(),

    "median_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].median(),

    "max_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].max(),

    "n_native_structural_features":len(structural_reduced)
}])

display(
    predictability_summary.round(4)
)

In [ ]:
# part B -- incremental predictive value
feature_universe = list(
    dict.fromkeys(
        lexical_plus_structural
    )
)

X = (
    df_all[
        feature_universe
    ]
    .copy()
)

y = (
    df_all["label"]
    .astype(int)
    .copy()
)

groups = (
    df_all["hostname"]
    .astype(str)
    .copy()
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

dev_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_dev = (
    X.iloc[dev_idx]
    .reset_index(drop=True)
)

X_test = (
    X.iloc[test_idx]
    .reset_index(drop=True)
)

y_dev = (
    y.iloc[dev_idx]
    .reset_index(drop=True)
)

y_test = (
    y.iloc[test_idx]
    .reset_index(drop=True)
)

groups_dev = (
    groups.iloc[dev_idx]
    .reset_index(drop=True)
)

groups_test = (
    groups.iloc[test_idx]
    .reset_index(drop=True)
)

print("Development samples:", len(X_dev))
print("Test samples:", len(X_test))

host_overlap = (
    set(groups_dev)
    &
    set(groups_test)
)

print(
    "Hostname overlap:",
    len(host_overlap)
)

assert len(host_overlap) == 0

In [ ]:
FIXED_XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 1.0
}


def make_xgb(seed=SEED):

    return XGBClassifier(
        **FIXED_XGB_PARAMS,
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1
    )


feature_sets = {
    "Lexical": lexical_reduced,
    "Structural": structural_reduced,
    "Lexical_plus_Structural": lexical_plus_structural,
}
for label, features in feature_sets.items():
    print(label, len(features))


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_splits = list(
    sgkf.split(
        X_dev,
        y_dev,
        groups=groups_dev
    )
)


def metric_dict(
    y_true,
    prob
):

    pred = (
        prob >= 0.5
    ).astype(int)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                pred
            )
    }


fold_records = []
oof_prediction_records = []

for set_name, features in (
    feature_sets.items()
):

    print(
        "\n"
        +
        "=" * 72
    )

    print(
        set_name
    )

    print(
        "=" * 72
    )

    oof_prob = np.full(
        len(X_dev),
        np.nan
    )

    for fold, (
        tr_idx,
        va_idx
    ) in enumerate(
        fold_splits,
        start=1
    ):

        model = make_xgb(
            SEED + fold
        )

        model.fit(
            X_dev[
                features
            ].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        prob = (
            model.predict_proba(
                X_dev[
                    features
                ].iloc[va_idx]
            )[:, 1]
        )

        oof_prob[va_idx] = prob

        metrics = metric_dict(
            y_dev.iloc[va_idx],
            prob
        )

        train_hosts = set(
            groups_dev.iloc[tr_idx]
        )

        val_hosts = set(
            groups_dev.iloc[va_idx]
        )

        metrics.update({
            "feature_set":
                set_name,

            "fold":
                fold,

            "hostname_overlap":
                len(
                    train_hosts
                    &
                    val_hosts
                )
        })

        fold_records.append(
            metrics
        )

        print(
            f"Fold {fold} | "
            f"F1={metrics['f1']:.4f} | "
            f"AUC={metrics['auc']:.4f} | "
            f"MCC={metrics['mcc']:.4f} | "
            f"overlap={metrics['hostname_overlap']}"
        )

    assert not np.isnan(
        oof_prob
    ).any()

    for i in range(
        len(X_dev)
    ):

        oof_prediction_records.append({
            "feature_set":
                set_name,

            "dev_row":
                i,

            "hostname":
                groups_dev.iloc[i],

            "label":
                int(
                    y_dev.iloc[i]
                ),

            "probability":
                float(
                    oof_prob[i]
                )
        })


fold_results = pd.DataFrame(
    fold_records
)

oof_predictions = pd.DataFrame(
    oof_prediction_records
)

display(
    fold_results.round(4)
)

In [ ]:
cv_summary = (
    fold_results
    .groupby(
        "feature_set"
    )
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean"
        ),
        std_accuracy=(
            "accuracy",
            "std"
        ),
        mean_precision=(
            "precision",
            "mean"
        ),
        mean_recall=(
            "recall",
            "mean"
        ),
        mean_f1=(
            "f1",
            "mean"
        ),
        std_f1=(
            "f1",
            "std"
        ),
        mean_auc=(
            "auc",
            "mean"
        ),
        std_auc=(
            "auc",
            "std"
        ),
        mean_mcc=(
            "mcc",
            "mean"
        ),
        std_mcc=(
            "mcc",
            "std"
        ),
        mean_hostname_overlap=(
            "hostname_overlap",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "mean_auc",
        ascending=False
    )
)

display(
    cv_summary.round(4)
)

In [ ]:
comparisons = [("Lexical_plus_Structural", "Lexical")]

delta_rows = []

for augmented, base in comparisons:

    aug = (
        fold_results[
            fold_results[
                "feature_set"
            ] == augmented
        ]
        .sort_values("fold")
        .reset_index(drop=True)
    )

    ref = (
        fold_results[
            fold_results[
                "feature_set"
            ] == base
        ]
        .sort_values("fold")
        .reset_index(drop=True)
    )

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        diffs = (
            aug[metric]
            -
            ref[metric]
        )

        delta_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "mean_delta":
                diffs.mean(),

            "std_delta":
                diffs.std(ddof=1),

            "wins_augmented":
                int(
                    (diffs > 0).sum()
                ),

            "losses_augmented":
                int(
                    (diffs < 0).sum()
                )
        })

delta_df = pd.DataFrame(
    delta_rows
)

display(
    delta_df.round(4)
)

In [ ]:
wilcoxon_rows = []

for augmented, base in comparisons:

    aug = (
        fold_results[
            fold_results[
                "feature_set"
            ] == augmented
        ]
        .sort_values("fold")
    )

    ref = (
        fold_results[
            fold_results[
                "feature_set"
            ] == base
        ]
        .sort_values("fold")
    )

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        x = (
            aug[metric]
            .to_numpy()
        )

        y0 = (
            ref[metric]
            .to_numpy()
        )

        try:

            stat, p = wilcoxon(
                x,
                y0,
                alternative="two-sided"
            )

        except ValueError:

            stat, p = (
                np.nan,
                1.0
            )

        wilcoxon_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "wilcoxon_stat":
                stat,

            "p_value":
                p
        })

wilcoxon_df = pd.DataFrame(
    wilcoxon_rows
)

display(
    wilcoxon_df.round(4)
)

print(
    "\nNote: with only five folds, "
    "Wilcoxon p-values have limited resolution. "
    "The grouped bootstrap below is therefore "
    "more informative for uncertainty."
)

In [ ]:
def get_oof_table(
    feature_set_name
):

    temp = (
        oof_predictions[
            oof_predictions[
                "feature_set"
            ]
            ==
            feature_set_name
        ]
        [
            [
                "dev_row",
                "hostname",
                "label",
                "probability"
            ]
        ]
        .copy()
        .sort_values(
            "dev_row"
        )
        .reset_index(drop=True)
    )

    return temp


def grouped_bootstrap_delta(
    augmented_name,
    base_name,
    n_boot=N_BOOT,
    seed=SEED
):

    aug = get_oof_table(
        augmented_name
    )

    ref = get_oof_table(
        base_name
    )

    assert np.array_equal(
        aug["dev_row"].values,
        ref["dev_row"].values
    )

    assert np.array_equal(
        aug["label"].values,
        ref["label"].values
    )

    hosts = (
        aug["hostname"]
        .unique()
    )

    host_to_rows = {
        h:
        np.where(
            aug["hostname"].values
            == h
        )[0]
        for h in hosts
    }

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for b in range(
        n_boot
    ):

        sampled_hosts = rng.choice(
            hosts,
            size=len(hosts),
            replace=True
        )

        sampled_indices = np.concatenate(
            [
                host_to_rows[h]
                for h in sampled_hosts
            ]
        )

        yb = (
            aug["label"]
            .values[
                sampled_indices
            ]
        )

        # Skip rare bootstrap draw with one class only.
        if len(
            np.unique(yb)
        ) < 2:
            continue

        aug_prob = (
            aug[
                "probability"
            ]
            .values[
                sampled_indices
            ]
        )

        ref_prob = (
            ref[
                "probability"
            ]
            .values[
                sampled_indices
            ]
        )

        aug_m = metric_dict(
            yb,
            aug_prob
        )

        ref_m = metric_dict(
            yb,
            ref_prob
        )

        rows.append({
            "delta_auc":
                aug_m["auc"]
                -
                ref_m["auc"],

            "delta_f1":
                aug_m["f1"]
                -
                ref_m["f1"],

            "delta_mcc":
                aug_m["mcc"]
                -
                ref_m["mcc"]
        })

    boot = pd.DataFrame(
        rows
    )

    summary_rows = []

    for metric in [
        "delta_auc",
        "delta_f1",
        "delta_mcc"
    ]:

        vals = boot[
            metric
        ].values

        summary_rows.append({
            "comparison":
                f"{augmented_name} - {base_name}",

            "metric":
                metric,

            "bootstrap_mean":
                np.mean(vals),

            "ci_2.5":
                np.percentile(
                    vals,
                    2.5
                ),

            "ci_97.5":
                np.percentile(
                    vals,
                    97.5
                ),

            "probability_delta_gt_0":
                np.mean(
                    vals > 0
                )
        })

    return (
        boot,
        pd.DataFrame(
            summary_rows
        )
    )


bootstrap_all = []
bootstrap_summaries = []

for augmented, base in comparisons:

    boot, summary = (
        grouped_bootstrap_delta(
            augmented,
            base
        )
    )

    boot[
        "comparison"
    ] = (
        f"{augmented} - {base}"
    )

    bootstrap_all.append(
        boot
    )

    bootstrap_summaries.append(
        summary
    )


bootstrap_df = pd.concat(
    bootstrap_all,
    ignore_index=True
)

bootstrap_summary = pd.concat(
    bootstrap_summaries,
    ignore_index=True
)

display(
    bootstrap_summary.round(4)
)

In [ ]:
holdout_rows = []

for set_name, features in (
    feature_sets.items()
):

    model = make_xgb(
        SEED
    )

    model.fit(
        X_dev[
            features
        ],
        y_dev
    )

    prob = (
        model.predict_proba(
            X_test[
                features
            ]
        )[:, 1]
    )

    metrics = metric_dict(
        y_test,
        prob
    )

    metrics.update({
        "feature_set":
            set_name,

        "hostname_overlap":
            len(
                set(groups_dev)
                &
                set(groups_test)
            )
    })

    holdout_rows.append(
        metrics
    )


holdout_df = (
    pd.DataFrame(
        holdout_rows
    )
    .sort_values(
        "auc",
        ascending=False
    )
)

display(
    holdout_df.round(4)
)

In [ ]:
holdout_delta_rows = []

holdout_indexed = (
    holdout_df
    .set_index(
        "feature_set"
    )
)

for augmented, base in comparisons:

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        holdout_delta_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "delta":
                holdout_indexed.loc[
                    augmented,
                    metric
                ]
                -
                holdout_indexed.loc[
                    base,
                    metric
                ]
        })

holdout_delta_df = pd.DataFrame(
    holdout_delta_rows
)

display(
    holdout_delta_df.round(4)
)

In [ ]:
# part C -- Exploratory Conditional Analysis
residual_records = []

for fold, (
    tr_idx,
    va_idx
) in enumerate(
    fold_splits,
    start=1
):

    Xlex_tr = (
        X_dev[
            lexical_reduced
        ]
        .iloc[tr_idx]
        .reset_index(drop=True)
    )

    Xlex_va = (
        X_dev[
            lexical_reduced
        ]
        .iloc[va_idx]
        .reset_index(drop=True)
    )

    S_tr = (
        X_dev[
            structural_intrinsic
        ]
        .iloc[tr_idx]
        .reset_index(drop=True)
    )

    S_va = (
        X_dev[
            structural_intrinsic
        ]
        .iloc[va_idx]
        .reset_index(drop=True)
    )

    reg = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=SEED + fold,
        n_jobs=-1
    )

    reg.fit(
        Xlex_tr,
        S_tr
    )

    S_hat_tr = reg.predict(
        Xlex_tr
    )

    S_hat_va = reg.predict(
        Xlex_va
    )

    R_tr = (
        S_tr.to_numpy()
        -
        S_hat_tr
    )

    R_va = (
        S_va.to_numpy()
        -
        S_hat_va
    )

    residual_names = [
        f"resid_{c}"
        for c in structural_intrinsic
    ]

    residual_tr_df = pd.DataFrame(
        R_tr,
        columns=residual_names
    )

    residual_va_df = pd.DataFrame(
        R_va,
        columns=residual_names
    )

    lexical_tr_df = (
        Xlex_tr
        .reset_index(drop=True)
    )

    lexical_va_df = (
        Xlex_va
        .reset_index(drop=True)
    )

    aug_tr = pd.concat(
        [
            lexical_tr_df,
            residual_tr_df
        ],
        axis=1
    )

    aug_va = pd.concat(
        [
            lexical_va_df,
            residual_va_df
        ],
        axis=1
    )

    lexical_model = make_xgb(
        SEED + fold
    )

    residual_model = make_xgb(
        SEED + fold
    )

    lexical_model.fit(
        lexical_tr_df,
        y_dev.iloc[tr_idx]
    )

    residual_model.fit(
        aug_tr,
        y_dev.iloc[tr_idx]
    )

    p_lex = (
        lexical_model.predict_proba(
            lexical_va_df
        )[:, 1]
    )

    p_resid = (
        residual_model.predict_proba(
            aug_va
        )[:, 1]
    )

    lex_m = metric_dict(
        y_dev.iloc[va_idx],
        p_lex
    )

    resid_m = metric_dict(
        y_dev.iloc[va_idx],
        p_resid
    )

    residual_records.append({
        "fold":
            fold,

        "lexical_auc":
            lex_m["auc"],

        "residual_augmented_auc":
            resid_m["auc"],

        "delta_auc":
            resid_m["auc"]
            -
            lex_m["auc"],

        "lexical_f1":
            lex_m["f1"],

        "residual_augmented_f1":
            resid_m["f1"],

        "delta_f1":
            resid_m["f1"]
            -
            lex_m["f1"],

        "lexical_mcc":
            lex_m["mcc"],

        "residual_augmented_mcc":
            resid_m["mcc"],

        "delta_mcc":
            resid_m["mcc"]
            -
            lex_m["mcc"]
    })


residual_analysis_df = pd.DataFrame(
    residual_records
)

display(
    residual_analysis_df.round(4)
)

In [ ]:
residual_summary = pd.DataFrame([{
    "mean_delta_auc":
        residual_analysis_df[
            "delta_auc"
        ].mean(),

    "std_delta_auc":
        residual_analysis_df[
            "delta_auc"
        ].std(ddof=1),

    "mean_delta_f1":
        residual_analysis_df[
            "delta_f1"
        ].mean(),

    "std_delta_f1":
        residual_analysis_df[
            "delta_f1"
        ].std(ddof=1),

    "mean_delta_mcc":
        residual_analysis_df[
            "delta_mcc"
        ].mean(),

    "std_delta_mcc":
        residual_analysis_df[
            "delta_mcc"
        ].std(ddof=1),

    "folds_auc_improved":
        int(
            (
                residual_analysis_df[
                    "delta_auc"
                ] > 0
            ).sum()
        ),

    "folds_f1_improved":
        int(
            (
                residual_analysis_df[
                    "delta_f1"
                ] > 0
            ).sum()
        )
}])

display(
    residual_summary.round(4)
)

In [ ]:
main_summary = (
    cv_summary[
        [
            "feature_set",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "std_auc",
            "mean_mcc",
            "std_mcc"
        ]
    ]
    .copy()
)

main_summary = (
    main_summary
    .merge(
        holdout_df[
            [
                "feature_set",
                "f1",
                "auc",
                "mcc"
            ]
        ]
        .rename(
            columns={
                "f1":
                    "holdout_f1",
                "auc":
                    "holdout_auc",
                "mcc":
                    "holdout_mcc"
            }
        ),
        on="feature_set",
        how="left"
    )
)

display(
    main_summary.round(4)
)

In [ ]:
OUT = Path(
    "experiment2_original_qr_corrected"
)

OUT.mkdir(
    exist_ok=True
)

cross_corr_df.to_csv(
    OUT / "cross_block_spearman.csv",
    index=False
)

cross_matrix.to_csv(
    OUT / "cross_block_spearman_matrix.csv"
)

structural_predictability_df.to_csv(
    OUT / "structural_predictability_from_lexical.csv",
    index=False
)

predictability_summary.to_csv(
    OUT / "structural_predictability_summary.csv",
    index=False
)

fold_results.to_csv(
    OUT / "representation_grouped_cv_fold_results.csv",
    index=False
)

cv_summary.to_csv(
    OUT / "representation_grouped_cv_summary.csv",
    index=False
)

oof_predictions.to_csv(
    OUT / "representation_oof_predictions.csv",
    index=False
)

delta_df.to_csv(
    OUT / "incremental_fold_deltas.csv",
    index=False
)

wilcoxon_df.to_csv(
    OUT / "paired_wilcoxon_results.csv",
    index=False
)

bootstrap_df.to_csv(
    OUT / "hostname_grouped_bootstrap_samples.csv",
    index=False
)

bootstrap_summary.to_csv(
    OUT / "hostname_grouped_bootstrap_summary.csv",
    index=False
)

holdout_df.to_csv(
    OUT / "representation_holdout_results.csv",
    index=False
)

holdout_delta_df.to_csv(
    OUT / "holdout_incremental_deltas.csv",
    index=False
)

residual_analysis_df.to_csv(
    OUT / "residualized_structure_fold_results.csv",
    index=False
)

residual_summary.to_csv(
    OUT / "residualized_structure_summary.csv",
    index=False
)

main_summary.to_csv(
    OUT / "paper_ready_representation_summary.csv",
    index=False
)

print(
    "Saved to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print(
        "-",
        p.name
    )
# Freeze the actual QR-native structural table used for every numerical result.
df_all[["original_index", "hostname", "label"] + structural_reduced].to_csv(OUT / "native_qr_structural_features.csv", index=False)


**Interpretation**: All QR matrices use the same fixed Version 13 and size. `qr_version` has no variance and is excluded. This experiment therefore asks whether **native QR pattern properties**, not re-encoding policy, add independent predictive utility. All three representation models use equal folds and fixed XGBoost parameters. Residualisation is an exploratory diagnostic only; it is not a causal proof of absence of QR-specific signal.

## Experiment 4 V2 — True hostname-disjoint held-out QR robustness

In [ ]:
import re
import cv2
import json
import joblib
import pickle
import warnings
from pathlib import Path
from urllib.parse import urlparse, urlunparse, urlsplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path(".")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

# Exact artifact names from the original final deployment notebook
FINAL_MODEL_PATH = Path("FINAL_10feature_XGB_for_deployment.pkl")
FINAL_CONFIG_PATH = Path("phase14B_final_robust_features.json")
DATA_PATH = Path("qr_full_feature_dataset_v2.csv")

OUT = Path("experiment4_holdout_corrected")
OUT.mkdir(exist_ok=True)

print("Model:", FINAL_MODEL_PATH.resolve())
print("Config:", FINAL_CONFIG_PATH.resolve())
print("Feature data:", DATA_PATH.resolve())

In [ ]:
with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

final_deployment_model = joblib.load(
    FINAL_MODEL_PATH
)

with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    deployment_config = json.load(f)

final_deployment_features = list(
    deployment_config["features"]
)

training_feature_data = pd.read_csv(
    DATA_PATH
)

required_cols = [
    "original_index",
    "url",
    "hostname",
    "label"
] + final_deployment_features

missing = [
    c for c in required_cols
    if c not in training_feature_data.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("QR shape:", qr_codes.shape)
print("Feature data shape:", training_feature_data.shape)
print("Final deployment features:")
for i, f in enumerate(final_deployment_features, 1):
    print(i, f)

In [ ]:
def normalize_url(url):
    url = str(url).strip()

    parse_url = url

    if not url.lower().startswith(
        ("http://", "https://")
    ):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    hostname = (
        parsed.hostname.lower()
        if parsed.hostname
        else ""
    )

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        port = parsed.port
    except ValueError:
        port = None

    netloc = hostname

    if port is not None:
        netloc += f":{port}"

    scheme = parsed.scheme.lower()

    normalized = urlunparse(
        (
            scheme,
            netloc,
            parsed.path,
            "",
            parsed.query,
            ""
        )
    )

    return normalized

In [ ]:
def ensure_scheme(url):

    url = str(url).strip()

    if not url:
        return ""

    has_real_scheme = re.match(
        r'^[A-Za-z][A-Za-z0-9+.\-]*://',
        url
    )

    if has_real_scheme:
        return url

    return "http://" + url


try:
    import tldextract

    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except ImportError:
    domain_extractor = None

# Frozen, ICANN-only public suffix fallback when tldextract is unavailable.
# The fallback has been checked against all 9713 frozen training samples,
# and the full feature consistency assertion below remains mandatory.
if domain_extractor is None:
    psl_file = DATA_DIR / "esann_icann_psl_snapshot.txt"
    if not psl_file.is_file():
        raise ImportError("Install tldextract or supply the frozen ICANN PSL snapshot")
    psl_rules = [s.strip() for s in psl_file.read_text(encoding="utf8").splitlines() if s.strip()]
    _PSL_EXACT = {s for s in psl_rules if not s.startswith(("*.","!"))}
    _PSL_WILD = {s[2:] for s in psl_rules if s.startswith("*.")}
    _PSL_EXCEPT = {s[1:] for s in psl_rules if s.startswith("!")}
    def offline_registered_domain(hostname):
        name = str(hostname).lower().strip(".")
        if not name: return name
        labels = name.split(".")
        public_suffix_size = 1
        for i in range(len(labels)):
            candidate = ".".join(labels[i:])
            if candidate in _PSL_EXCEPT:
                public_suffix_size=len(labels)-i-1
                break
            if candidate in _PSL_EXACT:
                public_suffix_size=max(public_suffix_size,len(labels)-i)
            if i < len(labels)-1 and ".".join(labels[i+1:]) in _PSL_WILD:
                public_suffix_size=max(public_suffix_size,len(labels)-i)
        return ".".join(labels[-(public_suffix_size+1):]) if len(labels)>public_suffix_size else name


def get_registered_domain(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return ""

    # Exact final compatibility fix from original notebook
    if hostname == "fed.us" or hostname.endswith(".fed.us"):
        return "fed.us"

    if domain_extractor is None:
        return offline_registered_domain(hostname)

    ext = domain_extractor(hostname)

    if not ext.domain:
        return hostname

    if ext.suffix:
        return ext.domain + "." + ext.suffix

    return ext.domain


def count_subdomains(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = get_registered_domain(
        hostname
    )

    hostname_parts = [
        x for x in hostname.split(".")
        if x
    ]

    domain_parts = [
        x for x in registered_domain.split(".")
        if x
    ]

    return max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )


def parse_deployment_url(url):

    raw_url = str(url).strip()

    normalized_url = normalize_url(
        raw_url
    )

    try:
        raw_parsed = urlsplit(
            ensure_scheme(raw_url)
        )

        raw_hostname = (
            raw_parsed.hostname
            or ""
        ).lower()

    except Exception:
        raw_hostname = ""

    try:
        normalized_parsed = urlsplit(
            normalized_url
        )

        normalized_path = (
            normalized_parsed.path
            or ""
        )

        normalized_query = (
            normalized_parsed.query
            or ""
        )

    except Exception:
        normalized_path = ""
        normalized_query = ""

    return {
        "raw_url": raw_url,
        "normalized_url": normalized_url,
        "raw_hostname": raw_hostname,
        "path": normalized_path,
        "query": normalized_query
    }

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_deployment_features(url):

    parsed = parse_deployment_url(
        url
    )

    raw_url = parsed["raw_url"]
    raw_hostname = parsed["raw_hostname"]
    path = parsed["path"]
    query = parsed["query"]

    url_lower = raw_url.lower()

    url_length = len(raw_url)

    # Exact training-compatible definition:
    # keep "www." in raw hostname length
    hostname_length = len(
        raw_hostname
    )

    # Exact training-compatible definition:
    # normalized path/query
    path_length = len(path)
    query_length = len(query)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_dots = raw_url.count(".")
    num_hyphens = raw_url.count("-")

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    num_subdomains = count_subdomains(
        raw_hostname
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio,

        "num_hyphens":
            num_hyphens,

        "query_length":
            query_length
    }


assert list(
    extract_deployment_features(
        "https://www.example.com/a?x=1"
    ).keys()
) == final_deployment_features

print("Exact final deployment extractor loaded.")

In [ ]:
consistency_records = []

for i, row in (
    training_feature_data
    .reset_index(drop=True)
    .iterrows()
):

    new_features = extract_deployment_features(
        row["url"]
    )

    for feature in final_deployment_features:

        old_value = float(
            row[feature]
        )

        new_value = float(
            new_features[feature]
        )

        consistency_records.append({
            "sample": i,
            "feature": feature,
            "old_value": old_value,
            "new_value": new_value,
            "difference":
                old_value - new_value
        })


deployment_consistency_df = pd.DataFrame(
    consistency_records
)

mismatch_df = deployment_consistency_df[
    deployment_consistency_df[
        "difference"
    ].abs() > 1e-12
].copy()

print("Samples checked:", len(training_feature_data))
print("Feature values checked:", len(deployment_consistency_df))
print(
    "Maximum absolute difference:",
    deployment_consistency_df[
        "difference"
    ].abs().max()
)
print("Mismatch count:", len(mismatch_df))

if len(mismatch_df) != 0:
    display(mismatch_df.head(30))
    raise RuntimeError(
        "STOP: deployment extractor no longer matches training features."
    )

print(
    "PASS: exact deployment consistency audit."
)
assert len(mismatch_df) == 0


In [ ]:
# Exact same split as final 10-feature model; NO sampling from development data.
base_df = training_feature_data[["original_index", "url", "hostname", "label"] + final_deployment_features].dropna(subset=["original_index","url","hostname","label"]).copy()
base_df["original_index"] = base_df["original_index"].astype(int)
base_df["label"] = base_df["label"].astype(int)
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
development_indices, holdout_indices = next(splitter.split(
    training_feature_data[final_deployment_features],
    training_feature_data["label"].astype(int),
    groups=training_feature_data["hostname"].astype(str)))
# Preserve exact row order from the final model's original GroupShuffleSplit.
eval_df = training_feature_data.iloc[holdout_indices][["original_index","url","hostname","label"] + final_deployment_features].copy().reset_index(drop=True)
dev_hosts = set(training_feature_data.iloc[development_indices].hostname.astype(str))
test_hosts = set(eval_df.hostname.astype(str))
assert len(dev_hosts & test_hosts) == 0, "CRITICAL: hostname leakage"
assert len(development_indices) == 7936 and len(eval_df) == 1777, "Split differs from frozen final model; STOP and investigate"
assert eval_df.original_index.is_unique
assert np.array_equal(labels[eval_df.original_index.astype(int).to_numpy()], eval_df.label.astype(int).to_numpy())
print("Correct independent holdout n:", len(eval_df))
print("Overlapping training/test hostnames:", len(dev_hosts & test_hosts))
print("Test label counts:", eval_df.label.value_counts().to_dict())


In [ ]:
direct_feature_records = []

for url in eval_df["url"]:
    direct_feature_records.append(
        extract_deployment_features(url)
    )

X_direct = pd.DataFrame(
    direct_feature_records
)[final_deployment_features]

y_eval = (
    eval_df["label"]
    .astype(int)
    .to_numpy()
)

direct_prob = (
    final_deployment_model
    .predict_proba(
        X_direct
    )[:, 1]
)

direct_pred = (
    direct_prob >= 0.5
).astype(int)


def metric_row(
    y_true,
    pred,
    prob
):

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                pred
            )
    }


direct_metrics = metric_row(
    y_eval,
    direct_pred,
    direct_prob
)

print("Direct-URL baseline on this exact sample:")
display(
    pd.DataFrame(
        [direct_metrics]
    ).round(4)
)
assert abs(direct_metrics["f1"] - 0.8810872027180068) < 1e-5, "Unexpected frozen model/split F1"
assert abs(direct_metrics["auc"] - 0.9509490182962518) < 1e-5, "Unexpected frozen model/split ROC-AUC"
assert abs(direct_metrics["mcc"] - 0.7641403806505308) < 1e-5, "Unexpected frozen model/split MCC"


In [ ]:
MODULE_SCALE = 10
QUIET_ZONE_MODULES = 4


def render_qr(qr_matrix):

    qr = np.asarray(
        qr_matrix
    )

    if qr.max() <= 1:
        # Exact orientation confirmed from original dataset:
        # 0 -> black, 1 -> white
        img = (
            qr * 255
        ).astype(np.uint8)

    else:
        img = qr.astype(
            np.uint8
        )

    img = cv2.copyMakeBorder(
        img,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        cv2.BORDER_CONSTANT,
        value=255
    )

    img = cv2.resize(
        img,
        None,
        fx=MODULE_SCALE,
        fy=MODULE_SCALE,
        interpolation=cv2.INTER_NEAREST
    )

    return img


detector = cv2.QRCodeDetector()


def decode_qr(img):

    data, _, _ = (
        detector.detectAndDecode(
            img
        )
    )

    if data is None:
        return None

    data = str(data).strip()

    if data == "":
        return None

    return data


def normalize_for_match(url):

    if url is None:
        return None

    try:
        return normalize_url(
            str(url)
        )
    except Exception:
        return str(url).strip()
# Parallel OpenCV decoders: no nested OpenCV threading. Individual image/payload procedures unchanged.
cv2.setNumThreads(1)


In [ ]:
# Decoding of all 1,777 original held-out QR codes occurs once under
# the Clean condition in the parallel evaluation cell below; do not duplicate it here.
print('Clean QR sanity check will use the exact full-holdout Clean condition.')

In [ ]:
# The clean direct-vs-decoded probability audit is derived after decoder evaluation.
# This keeps the clean and distortion populations identical and avoids double decoding.

In [ ]:
def apply_gaussian_blur(
    img,
    sigma
):

    k = max(
        3,
        int(
            2 * round(
                3 * sigma
            ) + 1
        )
    )

    if k % 2 == 0:
        k += 1

    return cv2.GaussianBlur(
        img,
        (k, k),
        sigmaX=sigma,
        sigmaY=sigma
    )


def rotate_bound_white(
    img,
    angle
):

    h, w = img.shape[:2]

    center = (
        w / 2,
        h / 2
    )

    M = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    cos = abs(
        M[0, 0]
    )

    sin = abs(
        M[0, 1]
    )

    new_w = int(
        h * sin
        +
        w * cos
    )

    new_h = int(
        h * cos
        +
        w * sin
    )

    M[0, 2] += (
        new_w / 2
        -
        center[0]
    )

    M[1, 2] += (
        new_h / 2
        -
        center[1]
    )

    return cv2.warpAffine(
        img,
        M,
        (
            new_w,
            new_h
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )


def apply_jpeg(
    img,
    quality
):

    ok, encoded = cv2.imencode(
        ".jpg",
        img,
        [
            int(
                cv2.IMWRITE_JPEG_QUALITY
            ),
            int(quality)
        ]
    )

    if not ok:
        raise RuntimeError(
            "JPEG encoding failed."
        )

    return cv2.imdecode(
        encoded,
        cv2.IMREAD_GRAYSCALE
    )


def apply_low_resolution(
    img,
    scale
):

    h, w = img.shape[:2]

    small = cv2.resize(
        img,
        (
            max(
                8,
                int(
                    round(
                        w * scale
                    )
                )
            ),
            max(
                8,
                int(
                    round(
                        h * scale
                    )
                )
            )
        ),
        interpolation=cv2.INTER_AREA
    )

    return cv2.resize(
        small,
        (
            w,
            h
        ),
        interpolation=cv2.INTER_LINEAR
    )


def apply_condition(
    img,
    kind,
    severity
):

    if kind == "clean":
        return img.copy()

    if kind == "blur":
        return apply_gaussian_blur(
            img,
            float(severity)
        )

    if kind == "rotation":
        return rotate_bound_white(
            img,
            float(severity)
        )

    if kind == "jpeg":
        return apply_jpeg(
            img,
            int(severity)
        )

    if kind == "scale":
        return apply_low_resolution(
            img,
            float(severity)
        )

    raise ValueError(kind)

In [ ]:
CONDITIONS = [
    ("clean", 0),

    ("blur", 0.5),
    ("blur", 1.0),
    ("blur", 1.5),

    ("rotation", -15),
    ("rotation", -10),
    ("rotation", -5),
    ("rotation", 5),
    ("rotation", 10),
    ("rotation", 15),

    ("jpeg", 80),
    ("jpeg", 50),
    ("jpeg", 30),

    ("scale", 0.75),
    ("scale", 0.50),
    ("scale", 0.25)
]


def condition_name(
    kind,
    severity
):

    if kind == "clean":
        return "Clean"

    if kind == "blur":
        return f"Blur_sigma_{severity}"

    if kind == "rotation":
        sign = (
            "plus"
            if float(
                severity
            ) > 0
            else "minus"
        )

        return (
            f"Rotation_{sign}_"
            f"{abs(float(severity))}deg"
        )

    if kind == "jpeg":
        return f"JPEG_Q{severity}"

    if kind == "scale":
        return f"Scale_{severity}"

    return (
        f"{kind}_{severity}"
    )

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading
import time

# Each worker owns a separate QRCodeDetector; no shared detector state.
THREADS = 4
_tls = threading.local()
def _decode_one_job(job):
    i, kind, severity = job
    row=eval_df.iloc[i]
    idx=int(row['original_index'])
    base=render_qr(qr_codes[idx])
    image=apply_condition(base,kind,severity)
    detector_obj=getattr(_tls,'detector',None)
    if detector_obj is None:
        detector_obj=cv2.QRCodeDetector()
        _tls.detector=detector_obj
    decoded,_,_=detector_obj.detectAndDecode(image)
    decoded=str(decoded).strip() if decoded is not None else ''
    return (i,decoded if decoded else None)

records=[]
for k,(kind,severity) in enumerate(CONDITIONS):
    t0=time.perf_counter()
    cname=condition_name(kind,severity)
    # The exact same 1,777 input rows are always used. Threads only speed up
    # independent decoding; transformations, detector and predictions are unchanged.
    with ThreadPoolExecutor(max_workers=THREADS) as pool:
        decoded=list(pool.map(_decode_one_job, ((i,kind,severity) for i in range(len(eval_df)))))
    condition_rows=[]
    for i, decoded_url in decoded:
        row=eval_df.iloc[i]
        truth=str(row['url'])
        correct=decoded_url is not None and normalize_for_match(decoded_url)==normalize_for_match(truth)
        condition_rows.append({
            'condition':cname,'kind':kind,'severity':severity,'sample_row':i,
            'original_index':int(row['original_index']),'hostname':str(row['hostname']),
            'true_label':int(row['label']),'true_url':truth,'decoded_url':decoded_url,
            'decode_output_success':decoded_url is not None,'correct_decode':bool(correct)
        })
    condition_df=pd.DataFrame(condition_rows)
    success_rows=condition_df.index[condition_df.decode_output_success].to_numpy()
    condition_df['predicted_label']=np.nan
    condition_df['predicted_probability']=np.nan
    if len(success_rows):
        Xdecoded=pd.DataFrame([extract_deployment_features(u) for u in condition_df.loc[success_rows,'decoded_url']])[final_deployment_features]
        probs=final_deployment_model.predict_proba(Xdecoded)[:,1]
        condition_df.loc[success_rows,'predicted_probability']=probs
        condition_df.loc[success_rows,'predicted_label']=(probs >= 0.5).astype(int)
    condition_df['classifier_correct'] = condition_df['predicted_label']==condition_df['true_label']
    condition_df['failure_mode']=np.select(
        [~condition_df.decode_output_success,~condition_df.correct_decode,~condition_df.classifier_correct],
        ['no_decode','decode_mismatch','classifier_error_after_correct_decode'],default='success'
    )
    records.extend(condition_df.to_dict(orient='records'))
    print(f'{k+1}/{len(CONDITIONS)} {cname}: correct decode {condition_df.correct_decode.mean():.5f}, {time.perf_counter()-t0:.1f} sec',flush=True)
    # Each condition is committed separately, so partial evaluations are auditable.
    condition_df.to_csv(OUT / f'decoding_{cname}.csv',index=False)
results_df=pd.DataFrame(records)
assert len(results_df)==len(eval_df)*len(CONDITIONS)
print('Total sample-condition pairs',len(results_df))

# Full 1,777-sample Clean sanity audit on same exact rows as all distortions.
clean_part=results_df[results_df.condition=='Clean'].sort_values('sample_row').reset_index(drop=True)
assert len(clean_part)==len(eval_df)
clean_audit_df=pd.DataFrame({
    'sample':clean_part.sample_row.astype(int),
    'decode_success':clean_part.decode_output_success.astype(bool),
    'correct_decode':clean_part.correct_decode.astype(bool),
    'direct_probability':direct_prob,
    'decoded_probability':clean_part.predicted_probability,
    'direct_prediction':direct_pred,
    'decoded_prediction':clean_part.predicted_label,
    'prediction_match': (clean_part.predicted_label.to_numpy()==direct_pred)
})
clean_audit_df['probability_abs_diff']=(clean_audit_df.direct_probability-clean_audit_df.decoded_probability).abs()
# Also verify new feature extraction on every correctly decoded clean payload.
feature_diffs=[]
for i,row in clean_part.iterrows():
    if row.correct_decode:
        expected=eval_df.iloc[i]
        extracted=extract_deployment_features(row.decoded_url)
        feature_diffs.append(max(abs(float(expected[f])-float(extracted[f])) for f in final_deployment_features))
    else:
        feature_diffs.append(np.nan)
clean_audit_df['feature_max_abs_diff']=feature_diffs
clean_summary=pd.DataFrame([{
    'n':len(clean_part), 'decode_output_rate':clean_audit_df.decode_success.mean(),
    'correct_decode_rate':clean_audit_df.correct_decode.mean(),
    'feature_mismatch_count':int((clean_audit_df.feature_max_abs_diff.fillna(np.inf)>1e-12).sum()),
    'max_feature_abs_diff':clean_audit_df.feature_max_abs_diff.max(),
    'prediction_mismatch_count':int((~clean_audit_df.prediction_match).sum()),
    'max_probability_abs_diff':clean_audit_df.probability_abs_diff.max()
}])
display(clean_summary)
if clean_summary.loc[0,'correct_decode_rate']<.99:
    raise RuntimeError('Unexpectedly poor clean QR decoding; inspect decoder and exact QR inputs')
if clean_summary.loc[0,'feature_mismatch_count'] or clean_summary.loc[0,'prediction_mismatch_count']:
    raise RuntimeError('Clean QR decoding and frozen direct URL pipeline do not match; investigate before any paper claims')
clean_decoded_prob=clean_audit_df.decoded_probability.to_numpy(dtype=float)
clean_decoded_pred=clean_audit_df.decoded_prediction.to_numpy(dtype=int)
clean_decoded_metrics=metric_row(y_eval,clean_decoded_pred,clean_decoded_prob)
sanity_metric_comparison=pd.DataFrame([{'pipeline':'Direct URL',**direct_metrics},{'pipeline':'Clean QR -> decoded URL',**clean_decoded_metrics}])
display(sanity_metric_comparison.round(6))


In [ ]:
def safe_auc(
    y_true,
    prob
):

    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob
    )


summary_rows = []

for cname, temp in (
    results_df.groupby(
        "condition",
        sort=False
    )
):

    n = len(temp)

    correct_decoded = temp[
        temp[
            "correct_decode"
        ]
    ].copy()

    if len(
        correct_decoded
    ) > 0:

        yt = (
            correct_decoded[
                "true_label"
            ]
            .astype(int)
            .to_numpy()
        )

        yp = (
            correct_decoded[
                "predicted_label"
            ]
            .astype(int)
            .to_numpy()
        )

        pr = (
            correct_decoded[
                "predicted_probability"
            ]
            .astype(float)
            .to_numpy()
        )

        cond = metric_row(
            yt,
            yp,
            pr
        )

    else:

        cond = {
            k: np.nan
            for k in [
                "accuracy",
                "precision",
                "recall",
                "f1",
                "auc",
                "mcc"
            ]
        }

    strict_e2e = (
        (
            temp[
                "correct_decode"
            ]
            &
            temp[
                "classifier_correct"
            ]
        ).mean()
    )

    phishing_mask = (
        temp[
            "true_label"
        ] == 1
    )

    e2e_phishing_capture = (
        (
            temp.loc[
                phishing_mask,
                "correct_decode"
            ]
            &
            (
                temp.loc[
                    phishing_mask,
                    "predicted_label"
                ] == 1
            )
        ).mean()
    )

    summary_rows.append({
        "condition":
            cname,

        "kind":
            temp[
                "kind"
            ].iloc[0],

        "severity":
            temp[
                "severity"
            ].iloc[0],

        "n":
            n,

        "decode_output_rate":
            temp[
                "decode_output_success"
            ].mean(),

        "correct_decode_rate":
            temp[
                "correct_decode"
            ].mean(),

        "decode_mismatch_rate":
            (
                temp[
                    "decode_output_success"
                ]
                &
                ~temp[
                    "correct_decode"
                ]
            ).mean(),

        "conditional_accuracy":
            cond["accuracy"],

        "conditional_precision":
            cond["precision"],

        "conditional_recall":
            cond["recall"],

        "conditional_f1":
            cond["f1"],

        "conditional_auc":
            cond["auc"],

        "conditional_mcc":
            cond["mcc"],

        "strict_e2e_correct_rate":
            strict_e2e,

        "end_to_end_phishing_capture":
            e2e_phishing_capture
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df.round(4)
)

In [ ]:
rotation_rows = summary_df[
    summary_df["kind"] == "rotation"
].copy()

rotation_rows[
    "abs_severity"
] = (
    rotation_rows[
        "severity"
    ]
    .astype(float)
    .abs()
)

rotation_aggregate = (
    rotation_rows
    .groupby(
        "abs_severity"
    )
    .agg(
        mean_correct_decode_rate=(
            "correct_decode_rate",
            "mean"
        ),
        min_correct_decode_rate=(
            "correct_decode_rate",
            "min"
        ),
        max_correct_decode_rate=(
            "correct_decode_rate",
            "max"
        ),
        mean_conditional_f1=(
            "conditional_f1",
            "mean"
        ),
        mean_conditional_auc=(
            "conditional_auc",
            "mean"
        ),
        mean_strict_e2e_correct_rate=(
            "strict_e2e_correct_rate",
            "mean"
        )
    )
    .reset_index()
)

display(
    rotation_aggregate.round(4)
)

In [ ]:
failure_counts = (
    results_df
    .groupby(
        [
            "condition",
            "failure_mode"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

failure_rates = (
    failure_counts
    .div(
        failure_counts.sum(
            axis=1
        ),
        axis=0
    )
)

display(
    failure_rates.round(4)
)

In [ ]:
clean = (
    summary_df[
        summary_df[
            "condition"
        ] == "Clean"
    ]
    .iloc[0]
)

retention_df = (
    summary_df.copy()
)

for metric in [
    "correct_decode_rate",
    "conditional_f1",
    "conditional_auc",
    "conditional_mcc",
    "strict_e2e_correct_rate",
    "end_to_end_phishing_capture"
]:

    retention_df[
        f"delta_{metric}_vs_clean"
    ] = (
        retention_df[
            metric
        ]
        -
        clean[
            metric
        ]
    )

display(
    retention_df.round(4)
)

In [ ]:
# Mutually exclusive failure attribution on the SAME complete holdout population.
# This is preferable to comparing percentage-point loss in decode rate with F1 loss.
bottleneck_rows = []
for condition, part in results_df.groupby("condition", sort=False):
    n = len(part)
    decode_fail = int((~part["correct_decode"].astype(bool)).sum())
    class_fail = int((part["correct_decode"].astype(bool) & ~part["classifier_correct"].astype(bool)).sum())
    success = int((part["correct_decode"].astype(bool) & part["classifier_correct"].astype(bool)).sum())
    assert decode_fail + class_fail + success == n
    total_fail = decode_fail + class_fail
    bottleneck_rows.append({
        "condition": condition,
        "n": n,
        "decode_or_payload_failure_count": decode_fail,
        "classifier_failure_after_correct_decode_count": class_fail,
        "end_to_end_success_count": success,
        "share_of_failures_due_to_decoding": decode_fail / total_fail if total_fail else np.nan,
        "share_of_failures_due_to_classifier": class_fail / total_fail if total_fail else np.nan,
    })
bottleneck_df = pd.DataFrame(bottleneck_rows)
display(bottleneck_df.round(4))


In [ ]:
clean_summary.to_csv(
    OUT / "clean_pipeline_sanity_summary.csv",
    index=False
)

clean_audit_df.to_csv(
    OUT / "clean_pipeline_sample_audit.csv",
    index=False
)

sanity_metric_comparison.to_csv(
    OUT / "direct_vs_clean_decoded_metrics.csv",
    index=False
)

results_df.to_csv(
    OUT / "sample_level_robustness_results.csv",
    index=False
)

summary_df.to_csv(
    OUT / "condition_level_summary.csv",
    index=False
)

rotation_aggregate.to_csv(
    OUT / "rotation_absolute_severity_summary.csv",
    index=False
)

failure_counts.to_csv(
    OUT / "failure_mode_counts.csv"
)

failure_rates.to_csv(
    OUT / "failure_mode_rates.csv"
)

retention_df.to_csv(
    OUT / "performance_retention_vs_clean.csv",
    index=False
)

bottleneck_df.to_csv(
    OUT / "bottleneck_analysis.csv",
    index=False
)

deployment_consistency_df.to_csv(
    OUT / "full_deployment_consistency_audit.csv",
    index=False
)

print(
    "Saved to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print(
        "-",
        p.name
    )
# Record EXACT sample identifiers and protocol for audit.
eval_df[["original_index","hostname","label"]].to_csv(OUT / "holdout_identifiers.csv", index=False)


### After execution: paper update checklist

1. Replace **all** structural statistics, Table 1 and structural-boostrap CIs with values from `experiment2_original_qr_corrected`.
2. Replace robustness clean/blur/rotation/JPEG/scale numbers with `experiment4_holdout_corrected/condition_level_summary.csv` and use the same held-out population for clean and distorted conditions.
3. Retain the frozen 10-feature XGBoost and its independent 1,777-sample holdout results, subject to the deployment-consistency assertion above.
4. Before any full reproduction or publication, unify the canonical 14-feature definition in the development notebook and recompute the paired 14-vs-10 bootstrap using the exact same pair of prediction arrays.
5. An external full-577,570 evaluation can be run separately after the corrected core experiments. Do not call model-only batch throughput a URL-to-decision throughput.

**Runtime constraint:** This notebook requires original pickles, the feature CSV and deployment model/config; notebook files alone are insufficient to recompute numerical results.